# 상승량 오차는 어느 박스에서 커졌을까?

무릎 통과와 바·몸 접촉에서 수동 허브, 수동 외곽 박스 중심, 실제 선택된 640 예측 박스 중심을 비교합니다. 수동 박스와 허브는 평가할 때만 사용합니다. 원본 영상에서 위로 이동한 거리를 양수로 셉니다.

In [ ]:
from datetime import datetime
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from wlcv.config import ROOT
from wlcv.evaluation.hub_rise_repeat import POINTS
from wlcv.evaluation.plate_rise_boxes import compare_box_rises, plot_components, show_b02

pairs = pd.read_csv(ROOT / "outputs/plate_resolution/2026-11-24/paired_frames.csv")
endpoints, intervals = compare_box_rises(pairs, pd.read_csv(POINTS))
output = ROOT / "outputs/plate_rise_boxes" / datetime.now().strftime("%Y%m%d-%H%M%S-%f")
output.mkdir(parents=True)
endpoints.to_csv(output / "endpoints.csv", index=False)
intervals.to_csv(output / "intervals.csv", index=False)

## 원본에서 두 끝점 보기

b02의 30번과 36번입니다. 금색 점선은 수동 외곽 박스, 파란 실선은 모델 박스, 흰 원은 1차 수동 허브입니다. 각 박스 중심과 아래·위 모서리가 어떻게 달라지는지 봅니다. `top_error`와 `bottom_error`는 모델 모서리 y에서 수동 모서리 y를 뺀 값으로, 양수면 모델 경계가 더 아래에 있습니다.

In [ ]:
fig = show_b02(pairs, endpoints)
fig.savefig(output / "b02_boxes.png", dpi=150)
display(fig)
plt.close(fig)
display(endpoints.loc[endpoints.person_id == "b02",
    ["event", "frame_index", "model_source", "top_error", "bottom_error", "model_minus_box_y"]].round(2))

## 같은 원본에서 세 상승량 대조

각 상승량은 무릎 통과 y에서 접촉 y를 뺀 값입니다. 모델−허브 오차는 **(모델 박스−수동 박스) + (수동 박스−수동 허브)**입니다. 서로 부호가 다르면 일부 상쇄됩니다. `first`는 6명의 첫 판독이고 `repeat`는 b02 한 사람의 재판독입니다. 각 원본의 픽셀 단위를 사용합니다.

In [ ]:
display(intervals[["person_id", "reading", "knee_frame", "contact_frame", "hub_rise", "box_rise",
                   "model_rise", "model_minus_box", "box_minus_hub", "model_minus_hub"]].round(2))
fig = plot_components(intervals)
fig.savefig(output / "rise_components.png", dpi=150)
display(fig)
plt.close(fig)
print(output)